# Healthcare Device Sales Prediction - Training & Evaluation

This notebook demonstrates the complete workflow for training and evaluating the PyTorch LSTM/Transformer models for predicting Abbott FreeStyle Libre sales based on diabetes prevalence data.

In [ ]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset

from data_preparation import DiabetesDataPreprocessor
from model import DiabetesSalesLSTM, DiabetesSalesTransformer
from train import train_epoch, validate, calculate_metrics
from utils import plot_training_history, plot_predictions, calculate_business_metrics

import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Data Preparation

In [ ]:
# Initialize preprocessor
preprocessor = DiabetesDataPreprocessor(
    s3_bucket='s3-data-lake',
    data_prefix='analytics/diabetes-sales'
)

# Load data from S3
print("Loading data from S3...")
df = preprocessor.load_data_from_s3()

print(f"\nDataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Engineer features
print("Engineering features...")
df = preprocessor.engineer_features(df)

print(f"\nFeatures created: {df.shape[1]} total columns")
print(f"\nNew features:")
new_features = ['sales_lag_1', 'sales_lag_2', 'sales_lag_3', 'sales_ma_3',
                'diabetes_growth', 'market_penetration', 'competition_score',
                'quarter_sin', 'quarter_cos', 'affordability_index']
df[new_features].describe()

In [ ]:
# Prepare sequences
print("Preparing sequences for LSTM...")
X, y = preprocessor.prepare_sequences(df, sequence_length=4, forecast_horizon=1)

print(f"\nInput shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTotal sequences: {len(X):,}")

In [ ]:
# Split and save data
print("Splitting data...")
preprocessor.split_and_save(X, y, test_size=0.2, val_size=0.1)
print("Data preparation complete!")

## 2. Model Training

In [ ]:
# Load processed data
X_train = torch.FloatTensor(np.load('/tmp/X_train.npy'))
y_train = torch.FloatTensor(np.load('/tmp/y_train.npy'))
X_val = torch.FloatTensor(np.load('/tmp/X_val.npy'))
y_val = torch.FloatTensor(np.load('/tmp/y_val.npy'))
X_test = torch.FloatTensor(np.load('/tmp/X_test.npy'))
y_test = torch.FloatTensor(np.load('/tmp/y_test.npy'))

print(f"Training samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print(f"Test samples: {len(X_test):,}")

In [ ]:
# Create data loaders
batch_size = 64

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
# Initialize LSTM model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = DiabetesSalesLSTM(
    input_size=13,
    hidden_size=128,
    num_layers=2,
    dropout=0.2
).to(device)

print(f"\nModel architecture:")
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Training setup
import torch.optim as optim
import torch.nn as nn
from model import EarlyStopping

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
early_stopping = EarlyStopping(patience=15, verbose=True)

print("Training configuration ready!")

In [ ]:
# Training loop
epochs = 50
history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': [], 'val_mape': []}

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_metrics = validate(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step(val_loss)
    
    # Log
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val MAE: {val_metrics['MAE']:.2f}")
    print(f"Val RMSE: {val_metrics['RMSE']:.2f}")
    print(f"Val MAPE: {val_metrics['MAPE']:.2f}%")
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(val_metrics['MAE'])
    history['val_rmse'].append(val_metrics['RMSE'])
    history['val_mape'].append(val_metrics['MAPE'])
    
    # Early stopping
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Early stopping triggered!")
        model.load_state_dict(early_stopping.best_model_state)
        break

print("\nTraining complete!")

## 3. Model Evaluation

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history['train_loss'], label='Train')
axes[0, 0].plot(history['val_loss'], label='Validation')
axes[0, 0].set_title('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(history['val_mae'])
axes[0, 1].set_title('Validation MAE')
axes[0, 1].grid(True)

axes[1, 0].plot(history['val_rmse'])
axes[1, 0].set_title('Validation RMSE')
axes[1, 0].grid(True)

axes[1, 1].plot(history['val_mape'])
axes[1, 1].set_title('Validation MAPE (%)')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Test set evaluation
test_loss, test_metrics = validate(model, test_loader, criterion, device)

print("\n" + "="*50)
print("TEST SET PERFORMANCE")
print("="*50)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test MAE: {test_metrics['MAE']:.2f} units")
print(f"Test RMSE: {test_metrics['RMSE']:.2f} units")
print(f"Test MAPE: {test_metrics['MAPE']:.2f}%")
print("="*50)

In [ ]:
# Get predictions
model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        predictions = model(X_batch).squeeze()
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(y_batch.numpy())

predictions = np.concatenate(all_predictions)
actuals = np.concatenate(all_targets)

print(f"Generated {len(predictions):,} predictions")

In [ ]:
# Plot predictions vs actuals
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter plot
axes[0].scatter(actuals, predictions, alpha=0.5)
axes[0].plot([actuals.min(), actuals.max()], [actuals.min(), actuals.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Sales (units)')
axes[0].set_ylabel('Predicted Sales (units)')
axes[0].set_title('Actual vs Predicted Sales')
axes[0].grid(True)

# Residuals
residuals = actuals - predictions
axes[1].scatter(predictions, residuals, alpha=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Sales (units)')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Business metrics
business_metrics = calculate_business_metrics(predictions, actuals, revenue_per_unit=45)

print("\n" + "="*50)
print("BUSINESS METRICS")
print("="*50)
for metric, value in business_metrics.items():
    if 'Revenue' in metric or 'Error' in metric:
        print(f"{metric}: ${value:,.2f}")
    else:
        print(f"{metric}: {value:.2f}")
print("="*50)

## 4. Make New Predictions

In [ ]:
# Example: Predict sales for a new country/quarter
# Create sample input (4 quarters of historical data)
sample_input = torch.FloatTensor([[
    [5.2, 1000000, 0.35, 8500, 8200, 8000, 8233, 0.015, 2.5, 0.42, 0.707, 0.707, 1.2],
    [5.3, 1005000, 0.36, 8700, 8500, 8200, 8467, 0.019, 2.6, 0.41, 0.0, 1.0, 1.25],
    [5.4, 1010000, 0.37, 8900, 8700, 8500, 8700, 0.019, 2.7, 0.40, -0.707, 0.707, 1.28],
    [5.5, 1015000, 0.38, 9100, 8900, 8700, 8900, 0.018, 2.8, 0.39, -1.0, 0.0, 1.30]
]]).to(device)

model.eval()
with torch.no_grad():
    prediction = model(sample_input).item()

print(f"\nPredicted sales for next quarter: {prediction:.2f} units")
print(f"Predicted revenue: ${prediction * 45:,.2f}")

## 5. Save Model

In [ ]:
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'test_metrics': test_metrics,
    'history': history
}, '../models/best_model.pth')

print("Model saved to ../models/best_model.pth")

## 6. Deploy to AWS SageMaker (Production)

### 6.1 Train on SageMaker

In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorch
from sagemaker import get_execution_role

# Initialize SageMaker session
sagemaker_session = sagemaker.Session()
role = get_execution_role()  # Or specify your IAM role ARN

print(f"SageMaker Role: {role}")
print(f"Default S3 Bucket: {sagemaker_session.default_bucket()}")
print(f"Region: {sagemaker_session.boto_region_name}")

# Define hyperparameters
hyperparameters = {
    's3-bucket': sagemaker_session.default_bucket(),
    's3-prefix': 'healthcare-sales-prediction',
    'model-type': 'lstm',
    'hidden-size': 128,
    'num-layers': 2,
    'dropout': 0.2,
    'epochs': 50,
    'batch-size': 64,
    'learning-rate': 0.001,
    'early-stopping-patience': 15
}

# Create PyTorch estimator
estimator = PyTorch(
    entry_point='train.py',
    source_dir='../src',
    role=role,
    framework_version='2.0.0',
    py_version='py310',
    instance_type='ml.m5.xlarge',
    instance_count=1,
    hyperparameters=hyperparameters,
    output_path=f's3://{sagemaker_session.default_bucket()}/models',
    sagemaker_session=sagemaker_session,
    enable_sagemaker_metrics=True,
    metric_definitions=[
        {'Name': 'train:loss', 'Regex': 'Train Loss: ([0-9\\.]+)'},
        {'Name': 'validation:loss', 'Regex': 'Val Loss: ([0-9\\.]+)'},
        {'Name': 'validation:mae', 'Regex': 'Val MAE: ([0-9\\.]+)'},
        {'Name': 'validation:rmse', 'Regex': 'Val RMSE: ([0-9\\.]+)'},
        {'Name': 'validation:mape', 'Regex': 'Val MAPE: ([0-9\\.]+)'}
    ],
    use_spot_instances=True,  # Cost savings!
    max_run=3600,  # 1 hour
    max_wait=7200   # 2 hours
)

print("\n✅ SageMaker estimator created!")
print("Instance type: ml.m5.xlarge (~$0.12/hour with spot instances)")
print("\nTo start training, run: estimator.fit(wait=True)")

In [ ]:
# Optional: Start training job (uncomment to run)
# estimator.fit(
#     job_name=f"diabetes-sales-lstm-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}",
#     wait=True,
#     logs='All'
# )

# After training completes, the model artifacts will be at:
# print(f"Model artifacts: {estimator.model_data}")

### 6.2 Deploy Model to SageMaker Endpoint

In [ ]:
from sagemaker.pytorch import PyTorchModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

# Option 1: Deploy model trained in this notebook (local)
# First, create model.tar.gz with your local model
# tar -czf model.tar.gz best_model.pth config.json
# aws s3 cp model.tar.gz s3://YOUR_BUCKET/models/

# Option 2: Use model from SageMaker training job
# model_data = estimator.model_data  # From previous cell

# For this demo, specify your model location
model_data = 's3://your-bucket/models/model.tar.gz'  # Update this!

# Create PyTorch model
pytorch_model = PyTorchModel(
    model_data=model_data,
    role=role,
    entry_point='inference.py',
    source_dir='../src',
    framework_version='2.0.0',
    py_version='py310',
    sagemaker_session=sagemaker_session
)

print("✅ PyTorchModel created!")
print(f"Model data: {model_data}")
print("\nReady to deploy to endpoint...")

In [ ]:
# Deploy to endpoint (uncomment to deploy)
# endpoint_name = 'diabetes-sales-lstm-prod'
# 
# predictor = pytorch_model.deploy(
#     endpoint_name=endpoint_name,
#     instance_type='ml.m5.large',
#     initial_instance_count=1,
#     serializer=JSONSerializer(),
#     deserializer=JSONDeserializer(),
#     wait=True
# )
# 
# print(f"✅ Model deployed to endpoint: {endpoint_name}")
# print(f"Cost: ~$0.115/hour (~$84/month if running 24/7)")
# print(f"With auto-scaling (8hrs/day): ~$28/month")

### 6.3 Make Predictions via SageMaker Endpoint

In [ ]:
# Connect to existing endpoint
from sagemaker.predictor import Predictor

endpoint_name = 'diabetes-sales-lstm-prod'  # Update with your endpoint name

# Create predictor
predictor = Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sagemaker_session,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer()
)

# Prepare input data (4 quarters, 13 features)
# Features: diabetes_prevalence, population, market_share, sales_lag_1, sales_lag_2, 
#           sales_lag_3, sales_ma_3, diabetes_growth, market_penetration, 
#           competition_score, quarter_sin, quarter_cos, affordability_index

sample_input = {
    "sequences": [
        [
            # Q1 2024
            [5.2, 1000000, 0.35, 8500, 8200, 8000, 8233, 0.015, 2.5, 0.42, 0.707, 0.707, 1.2],
            # Q2 2024
            [5.3, 1005000, 0.36, 8700, 8500, 8200, 8467, 0.019, 2.6, 0.41, 0.0, 1.0, 1.25],
            # Q3 2024
            [5.4, 1010000, 0.37, 8900, 8700, 8500, 8700, 0.019, 2.7, 0.40, -0.707, 0.707, 1.28],
            # Q4 2024
            [5.5, 1015000, 0.38, 9100, 8900, 8700, 8900, 0.018, 2.8, 0.39, -1.0, 0.0, 1.30]
        ]
    ]
}

# Make prediction
try:
    response = predictor.predict(sample_input)
    
    print("✅ Prediction successful!")
    print(f"\nPredicted sales for Q1 2025: {response['predictions'][0]:,.2f} units")
    print(f"Estimated revenue (€45/unit): €{response['predictions'][0] * 45:,.2f}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure the endpoint is deployed and running!")

### 6.4 Batch Predictions with SageMaker

In [ ]:
# Batch predictions for multiple countries
countries_data = {
    'Germany': [
        [7.2, 83000000, 0.42, 12000, 11800, 11500, 11767, 0.018, 3.2, 0.38, 0.707, 0.707, 1.5],
        [7.3, 83100000, 0.43, 12200, 12000, 11800, 12000, 0.014, 3.3, 0.37, 0.0, 1.0, 1.52],
        [7.4, 83200000, 0.44, 12400, 12200, 12000, 12200, 0.016, 3.4, 0.36, -0.707, 0.707, 1.54],
        [7.5, 83300000, 0.45, 12600, 12400, 12200, 12400, 0.014, 3.5, 0.35, -1.0, 0.0, 1.56]
    ],
    'France': [
        [6.8, 67000000, 0.38, 9500, 9300, 9100, 9300, 0.021, 2.9, 0.41, 0.707, 0.707, 1.3],
        [6.9, 67100000, 0.39, 9700, 9500, 9300, 9500, 0.015, 3.0, 0.40, 0.0, 1.0, 1.32],
        [7.0, 67200000, 0.40, 9900, 9700, 9500, 9700, 0.015, 3.1, 0.39, -0.707, 0.707, 1.34],
        [7.1, 67300000, 0.41, 10100, 9900, 9700, 9900, 0.014, 3.2, 0.38, -1.0, 0.0, 1.36]
    ],
    'Spain': [
        [5.8, 47000000, 0.32, 7200, 7000, 6800, 7000, 0.029, 2.6, 0.44, 0.707, 0.707, 1.1],
        [5.9, 47100000, 0.33, 7400, 7200, 7000, 7200, 0.017, 2.7, 0.43, 0.0, 1.0, 1.12],
        [6.0, 47200000, 0.34, 7600, 7400, 7200, 7400, 0.017, 2.8, 0.42, -0.707, 0.707, 1.14],
        [6.1, 47300000, 0.35, 7800, 7600, 7400, 7600, 0.016, 2.9, 0.41, -1.0, 0.0, 1.16]
    ]
}

# Make batch predictions
batch_input = {
    "sequences": list(countries_data.values())
}

try:
    batch_response = predictor.predict(batch_input)
    
    print("✅ Batch predictions successful!\n")
    print("=" * 70)
    print(f"{'Country':<15} {'Predicted Sales':<20} {'Est. Revenue (€45/unit)'}")
    print("=" * 70)
    
    for i, country in enumerate(countries_data.keys()):
        predicted_sales = batch_response['predictions'][i]
        revenue = predicted_sales * 45
        print(f"{country:<15} {predicted_sales:>15,.0f} units {revenue:>20,.2f} €")
    
    total_sales = sum(batch_response['predictions'])
    total_revenue = total_sales * 45
    
    print("=" * 70)
    print(f"{'TOTAL':<15} {total_sales:>15,.0f} units {total_revenue:>20,.2f} €")
    print("=" * 70)
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure the endpoint is deployed and running!")

### 6.5 Enable Auto-Scaling for Production

In [ ]:
import boto3

# Configure auto-scaling for cost optimization
autoscaling_client = boto3.client('application-autoscaling')

endpoint_name = 'diabetes-sales-lstm-prod'
resource_id = f'endpoint/{endpoint_name}/variant/AllTraffic'

# Register scalable target
autoscaling_client.register_scalable_target(
    ServiceNamespace='sagemaker',
    ResourceId=resource_id,
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    MinCapacity=1,
    MaxCapacity=3
)

# Define scaling policy (target tracking)
autoscaling_client.put_scaling_policy(
    PolicyName=f'{endpoint_name}-scaling-policy',
    ServiceNamespace='sagemaker',
    ResourceId=resource_id,
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    PolicyType='TargetTrackingScaling',
    TargetTrackingScalingPolicyConfiguration={
        'TargetValue': 70.0,  # Target 70 invocations per instance
        'PredefinedMetricSpecification': {
            'PredefinedMetricType': 'SageMakerVariantInvocationsPerInstance'
        },
        'ScaleInCooldown': 300,   # 5 minutes
        'ScaleOutCooldown': 60     # 1 minute
    }
)

print("Auto-scaling configured!")
print(f"Min instances: 1")
print(f"Max instances: 3")
print(f"Target: 70 invocations/instance")
print(f"\nCost savings: Scales down during low traffic (saves ~66% during off-hours)")

### 6.6 Cleanup (Delete Endpoint to Save Costs)

In [ ]:
# Delete endpoint when no longer needed (uncomment to delete)
# sagemaker_client = boto3.client('sagemaker')
# 
# endpoint_name = 'diabetes-sales-lstm-prod'
# 
# try:
#     sagemaker_client.delete_endpoint(EndpointName=endpoint_name)
#     print(f"Endpoint '{endpoint_name}' deleted successfully!")
#     print("No more charges will be incurred.")
# except Exception as e:
#     print(f"Error deleting endpoint: {e}")

print("Delete endpoints when not in use to avoid charges!")
print("Cost: ml.m5.large = ~$0.115/hour = ~$84/month if left running 24/7")